In [2]:
import pandas as pd
import numpy as np

# Les 9 tables d'Olist
orders     = pd.read_csv("data/olist_orders_dataset.csv")
items      = pd.read_csv("data/olist_order_items_dataset.csv")
reviews    = pd.read_csv("data/olist_order_reviews_dataset.csv")
customers  = pd.read_csv("data/olist_customers_dataset.csv")
sellers    = pd.read_csv("data/olist_sellers_dataset.csv")
products   = pd.read_csv("data/olist_products_dataset.csv")
payments   = pd.read_csv("data/olist_order_payments_dataset.csv")
geo        = pd.read_csv("data/olist_geolocation_dataset.csv")
traduction = pd.read_csv("data/product_category_name_translation.csv")

# Les 5 colonnes de dates : texte -> vraies dates (comme au TP1)
colonnes_dates = ["order_purchase_timestamp", "order_approved_at",
                  "order_delivered_carrier_date", "order_delivered_customer_date",
                  "order_estimated_delivery_date"]
for col in colonnes_dates:
    orders[col] = pd.to_datetime(orders[col])

print("Période :", orders["order_purchase_timestamp"].min(), "->", orders["order_purchase_timestamp"].max())
print(orders["order_status"].value_counts())

Période : 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [3]:
tables = {"orders": orders, "items": items, "reviews": reviews, "customers": customers,
          "sellers": sellers, "products": products, "payments": payments, "geo": geo}
for nom, table in tables.items():
    manquants = table.isna().sum()
    manquants = manquants[manquants > 0]
    print(f"--- {nom} : {len(table)} lignes, {table.duplicated().sum()} lignes en double")
    if len(manquants) > 0:
        print(f"    valeurs manquantes : {manquants.to_dict()}")

print("\norder_id en double dans reviews :", reviews["order_id"].duplicated().sum())

--- orders : 99441 lignes, 0 lignes en double
    valeurs manquantes : {'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965}
--- items : 112650 lignes, 0 lignes en double
--- reviews : 99224 lignes, 0 lignes en double
    valeurs manquantes : {'review_comment_title': 87656, 'review_comment_message': 58247}
--- customers : 99441 lignes, 0 lignes en double
--- sellers : 3095 lignes, 0 lignes en double
--- products : 32951 lignes, 0 lignes en double
    valeurs manquantes : {'product_category_name': 610, 'product_name_lenght': 610, 'product_description_lenght': 610, 'product_photos_qty': 610, 'product_weight_g': 2, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2}
--- payments : 103886 lignes, 0 lignes en double
--- geo : 1000163 lignes, 261831 lignes en double

order_id en double dans reviews : 551


In [4]:
livrees = orders[(orders["order_status"] == "delivered")
                 & orders["order_delivered_customer_date"].notna()].copy()
livrees = livrees[(livrees["order_purchase_timestamp"] >= "2017-01-01")
                  & (livrees["order_purchase_timestamp"] < "2018-09-01")]
incoherent = ((livrees["order_delivered_carrier_date"] < livrees["order_purchase_timestamp"])
              | (livrees["order_delivered_customer_date"] < livrees["order_delivered_carrier_date"])
              | livrees["order_delivered_carrier_date"].isna())
livrees = livrees[~incoherent]
print("Commandes gardées :", len(livrees), f"({len(livrees) / len(orders) * 100:.1f} %)")

Commandes gardées : 96018 (96.6 %)


In [5]:
un_jour = pd.Timedelta(days=1)
livrees["delai_livraison_j"]   = (livrees["order_delivered_customer_date"] - livrees["order_purchase_timestamp"]) / un_jour
livrees["delai_preparation_j"] = (livrees["order_delivered_carrier_date"]  - livrees["order_purchase_timestamp"]) / un_jour
livrees["delai_transport_j"]   = (livrees["order_delivered_customer_date"] - livrees["order_delivered_carrier_date"]) / un_jour
livrees["delai_promis_j"]      = (livrees["order_estimated_delivery_date"] - livrees["order_purchase_timestamp"]) / un_jour
livrees["retard_j"]  = (livrees["order_delivered_customer_date"].dt.normalize()
                        - livrees["order_estimated_delivery_date"]).dt.days
livrees["en_retard"] = livrees["retard_j"] > 0

print(f"Délai de livraison moyen : {livrees['delai_livraison_j'].mean():.1f} jours (médiane {livrees['delai_livraison_j'].median():.1f})")
print(f"  préparation vendeur    : {livrees['delai_preparation_j'].mean():.1f} jours")
print(f"  transport              : {livrees['delai_transport_j'].mean():.1f} jours")
print(f"Délai promis moyen       : {livrees['delai_promis_j'].mean():.1f} jours")
print(f"Taux de retard           : {livrees['en_retard'].mean() * 100:.1f} %  ({livrees['en_retard'].sum()} commandes)")
print(f"Livraisons > 60 jours    : {(livrees['delai_livraison_j'] > 60).sum()}")

Délai de livraison moyen : 12.5 jours (médiane 10.2)
  préparation vendeur    : 3.2 jours
  transport              : 9.3 jours
Délai promis moyen       : 23.7 jours
Taux de retard           : 6.8 %  (6528 commandes)
Livraisons > 60 jours    : 303


In [6]:
# ÉTAPE 4 : retard et satisfaction client

# 1) Un seul avis par commande : on garde le plus récent
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
avis = (reviews.sort_values("review_creation_date")
               .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])

# 2) Jointure commandes + avis (comme un JOIN en SQL)
livrees = livrees.merge(avis, on="order_id", how="left")
print("Commandes avec une note :", livrees["review_score"].notna().sum())

# 3) Note moyenne : à l'heure vs en retard
print("\nNote moyenne :")
print(livrees.groupby("en_retard")["review_score"].mean().round(2))

# 4) Plus le retard est long, plus la note baisse ?
livrees["classe_retard"] = pd.cut(livrees["retard_j"], bins=[-1000, 0, 3, 7, 14, 1000],
                                  labels=["À l'heure", "1 à 3 j", "4 à 7 j", "8 à 14 j", "Plus de 14 j"])
tableau = livrees.groupby("classe_retard", observed=True).agg(
    commandes=("order_id", "size"),
    note_moyenne=("review_score", "mean"),
    part_notes_1_2=("review_score", lambda s: (s <= 2).sum() / s.notna().sum() * 100),
).round(1)
tableau

Commandes avec une note : 95376

Note moyenne :
en_retard
False    4.29
True     2.27
Name: review_score, dtype: float64


,commandes,note_moyenne,part_notes_1_2
classe_retard,,,
À l'heure,89490,4.3,9.2
1 à 3 j,1868,3.3,32.2
4 à 7 j,1800,2.1,67.6
8 à 14 j,1478,1.7,80.2
Plus de 14 j,1382,1.7,78.3


In [7]:
# Exemple : le calcul détaillé pour UNE commande en retard
exemple = livrees[livrees["en_retard"]].iloc[0]

print("Achat                   :", exemple["order_purchase_timestamp"])
print("Remise au transporteur  :", exemple["order_delivered_carrier_date"])
print("Livraison au client     :", exemple["order_delivered_customer_date"])
print("Date promise            :", exemple["order_estimated_delivery_date"])
print()
print(f"Préparation vendeur = remise - achat         = {exemple['delai_preparation_j']:.1f} jours")
print(f"Transport           = livraison - remise     = {exemple['delai_transport_j']:.1f} jours")
print(f"Délai total         = livraison - achat      = {exemple['delai_livraison_j']:.1f} jours")
print(f"Retard              = jour livré - jour promis = {exemple['retard_j']} jours")
print(f"Note du client      : {exemple['review_score']}")

Achat                   : 2017-09-18 14:31:30
Remise au transporteur  : 2017-10-06 17:50:03
Livraison au client     : 2017-10-09 22:23:46
Date promise            : 2017-09-28 00:00:00

Préparation vendeur = remise - achat         = 18.1 jours
Transport           = livraison - remise     = 3.2 jours
Délai total         = livraison - achat      = 21.3 jours
Retard              = jour livré - jour promis = 11 jours
Note du client      : 2.0


In [8]:
import plotly.graph_objects as go

# Couleurs : une seule couleur d'accent (le retard), le reste en gris
ROUGE = "#B4483F"      # retard / problème
BLEU = "#1B6A8F"       # à l'heure
GRIS = "#9CA3AF"
GRIS_FONCE = "#4B5563"

def fr(x, dec=1):
    """Format français : espace pour les milliers, virgule pour les décimales."""
    return f"{x:,.{dec}f}".replace(",", " ").replace(".", ",")

def mise_en_forme(fig, titre, sous_titre=None, hauteur=450):
    """Style commun (comme au TP1) : titre-message + sous-titre, fond blanc, largeur fixe."""
    texte = f"<b>{titre}</b>"
    if sous_titre:
        texte += f"<br><span style='font-size:13px;color:#6B7280'>{sous_titre}</span>"
    fig.update_layout(template="simple_white", width=900, height=hauteur, separators=", ",
                      title=dict(text=texte, x=0.02, xanchor="left", font=dict(size=18)),
                      font=dict(family="Arial", size=13, color="#1F2937"),
                      margin=dict(t=100, l=60, r=30, b=50), showlegend=False)
    return fig

In [9]:
# Q1 : les indicateurs clés, chacun dans sa carte
n             = len(livrees)
n_retard      = int(livrees["en_retard"].sum())
taux_retard   = n_retard / n * 100
note_heure    = livrees.loc[~livrees["en_retard"], "review_score"].mean()
note_retard   = livrees.loc[livrees["en_retard"], "review_score"].mean()
delai_moyen   = livrees["delai_livraison_j"].mean()
delai_promis  = livrees["delai_promis_j"].mean()
notes_semaine = livrees.loc[livrees["retard_j"] > 7, "review_score"]          # plus d'une semaine de retard
part_mauvaises = (notes_semaine <= 2).sum() / notes_semaine.notna().sum() * 100

# Une carte = une étiquette, une valeur, une couleur, un détail
cartes = [
    {"etiquette": "COMMANDES LIVRÉES",   "valeur": fr(n, 0),             "couleur": GRIS_FONCE,
     "detail": "janv. 2017 → août 2018<br>20 mois"},
    {"etiquette": "DÉLAI DE LIVRAISON",  "valeur": f"{fr(delai_moyen)} j", "couleur": BLEU,
     "detail": f"en moyenne<br>promis : {fr(delai_promis)} j"},
    {"etiquette": "COMMANDES EN RETARD", "valeur": f"{fr(taux_retard)} %", "couleur": ROUGE,
     "detail": f"{fr(n_retard, 0)} commandes<br>livrées après la date promise"},
    {"etiquette": "NOTE SI RETARD",      "valeur": f"{fr(note_retard)} / 5", "couleur": ROUGE,
     "detail": f"contre {fr(note_heure)} à l'heure<br>{fr(part_mauvaises, 0)} % de 1–2 ★ après 1 semaine"},
]

fig = go.Figure()
largeur, ecart = 0.235, 0.02                      # largeur d'une carte et espace entre les cartes
for i, c in enumerate(cartes):
    x0 = i * (largeur + ecart)
    x1 = x0 + largeur
    # Fond gris clair de la carte
    fig.add_shape(type="rect", xref="paper", yref="paper", x0=x0, x1=x1, y0=0, y1=1,
                  fillcolor="#F9FAFB", line=dict(color="#E5E7EB", width=1), layer="below")
    # Bande de couleur en haut de la carte
    fig.add_shape(type="rect", xref="paper", yref="paper", x0=x0, x1=x1, y0=0.965, y1=1,
                  fillcolor=c["couleur"], line=dict(width=0), opacity=1, layer="above")
    # Textes : étiquette, valeur, détail
    fig.add_annotation(x=x0 + 0.02, y=0.84, text=c["etiquette"], font=dict(size=11, color="#6B7280"))
    fig.add_annotation(x=x0 + 0.02, y=0.55, text=f"<b>{c['valeur']}</b>", font=dict(size=34, color=c["couleur"]))
    fig.add_annotation(x=x0 + 0.02, y=0.17, text=c["detail"], align="left", font=dict(size=12, color=GRIS_FONCE))

fig.update_annotations(xref="paper", yref="paper", xanchor="left", showarrow=False)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
mise_en_forme(fig, "Le retard est rare, mais il divise la note client par deux",
              f"Indicateurs clés de livraison · {fr(n, 0)} commandes livrées", hauteur=290)
fig.update_layout(margin=dict(t=95, l=20, r=20, b=15))
fig.show()

In [10]:
# Q2 : part des clients mécontents (notes 1 ou 2) selon la durée du retard
t = tableau.reset_index()          # tableau calculé à l'étape 4 (une ligne par classe de retard)

# Couleurs : bleu = à l'heure, gris = petit retard, rouge = retard de 4 jours et plus
couleurs = []
for classe in t["classe_retard"]:
    if classe == "À l'heure":
        couleurs.append(BLEU)
    elif classe == "1 à 3 j":
        couleurs.append(GRIS)
    else:
        couleurs.append(ROUGE)

fig = go.Figure(go.Bar(
    x=t["classe_retard"].astype(str), y=t["part_notes_1_2"], marker_color=couleurs,
    text=[f"<b>{fr(v, 0)} %</b>" for v in t["part_notes_1_2"]], textposition="outside", textfont=dict(size=15),
    customdata=t[["commandes", "note_moyenne"]],
    hovertemplate="%{x}<br>%{y:.0f} % de notes 1–2 ★<br>note moyenne %{customdata[1]:.1f}"
                  "<br>%{customdata[0]:,} commandes<extra></extra>"))

# Nombre de commandes sous chaque barre (pour montrer que les groupes sont assez grands)
for _, ligne in t.iterrows():
    fig.add_annotation(x=str(ligne["classe_retard"]), y=-9, text=f"{fr(ligne['commandes'], 0)} cmd",
                       showarrow=False, font=dict(size=11, color=GRIS))

fig.update_yaxes(visible=False, range=[-14, 95])          # pas d'axe : les valeurs sont écrites sur les barres
fig.update_xaxes(title=None, showline=False, ticks="", tickfont=dict(size=13))
mise_en_forme(fig, "Dès 4 jours de retard, deux clients sur trois sont mécontents",
              "Part des commandes notées 1 ou 2 étoiles, selon le retard par rapport à la date promise",
              hauteur=430)
fig.update_layout(bargap=0.35)
fig.show()

In [11]:
# Q3 : quand les retards arrivent-ils ? (évolution par semaine)
from plotly.subplots import make_subplots

# 1) Regrouper les commandes par semaine d'achat
semaines = (livrees.set_index("order_purchase_timestamp")
                   .resample("W-MON", label="left", closed="left")          # semaines du lundi au dimanche
                   .agg(commandes=("order_id", "size"), taux_retard=("en_retard", "mean")))
semaines["taux_retard"] = semaines["taux_retard"] * 100
semaines = semaines.iloc[1:-1]          # on enlève la 1re et la dernière semaine (incomplètes)

# 2) Deux graphiques l'un sous l'autre, avec le même axe des dates (pas de double axe Y)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.4, 0.6], vertical_spacing=0.08)

# En haut : le volume de commandes (gris = contexte)
fig.add_trace(go.Scatter(x=semaines.index, y=semaines["commandes"], mode="lines",
                         line=dict(color=GRIS_FONCE, width=1.5),
                         fill="tozeroy", fillcolor="rgba(156,163,175,0.25)",
                         hovertemplate="Semaine du %{x|%d/%m/%Y}<br>%{y} commandes<extra></extra>"),
              row=1, col=1)

# En bas : le taux de retard (rouge = le message)
fig.add_trace(go.Scatter(x=semaines.index, y=semaines["taux_retard"], mode="lines",
                         line=dict(color=ROUGE, width=2.5),
                         hovertemplate="Semaine du %{x|%d/%m/%Y}<br>%{y:.1f} % en retard<extra></extra>"),
              row=2, col=1)

# Ligne de la moyenne
moyenne = livrees["en_retard"].mean() * 100
fig.add_hline(y=moyenne, line=dict(color=GRIS, dash="dot", width=1), row=2, col=1)
fig.add_annotation(x=semaines.index[1], y=moyenne, yshift=10, xanchor="left", showarrow=False,
                   text=f"moyenne {fr(moyenne)} %", font=dict(size=11, color=GRIS), row=2, col=1)

# 3) Les deux périodes à problème, en rouge très clair sur les deux graphiques
for debut, fin in [("2017-11-20", "2017-12-04"), ("2018-02-05", "2018-03-26")]:
    for ligne in (1, 2):
        fig.add_vrect(x0=debut, x1=fin, fillcolor=ROUGE, opacity=0.08, line_width=0, row=ligne, col=1)

fig.add_annotation(x="2017-11-27", y=semaines["commandes"].max(), yshift=8, yanchor="bottom", showarrow=False,
                   text="<b>Black Friday</b><br>24 nov. 2017", font=dict(size=11, color=ROUGE), row=1, col=1)
fig.add_annotation(x="2018-03-01", y=semaines["taux_retard"].max(), xanchor="left", xshift=50, showarrow=False,
                   text="<b>Février – mars 2018</b><br>jusqu'à 1 commande sur 4 en retard",
                   font=dict(size=11, color=ROUGE), row=2, col=1)

# 4) Axes : mois en français
MOIS = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
reperes = pd.date_range("2017-01-01", "2018-08-01", freq="2MS")             # un repère tous les 2 mois
fig.update_xaxes(tickvals=reperes, ticktext=[f"{MOIS[d.month - 1]} {d.year}" for d in reperes], row=2, col=1)
fig.update_yaxes(title=dict(text="Commandes / semaine", font=dict(size=12)),
                 range=[0, semaines["commandes"].max() * 1.35], row=1, col=1)
fig.update_yaxes(title=dict(text="% en retard", font=dict(size=12)), ticksuffix=" %", rangemode="tozero", row=2, col=1)

mise_en_forme(fig, "Les retards explosent après Black Friday et en février – mars 2018",
              "Par semaine d'achat · en haut : volume de commandes · en bas : part des commandes livrées en retard",
              hauteur=560)
fig.show()

In [12]:
# Q4 : où les retards arrivent-ils ? (état du vendeur x état du client)

# 1) Les 5 grandes régions du Brésil
REGIONS = {"Sud-Est":      ["ES", "MG", "RJ", "SP"],
           "Sud":          ["PR", "RS", "SC"],
           "Centre-Ouest": ["DF", "GO", "MS", "MT"],
           "Nord-Est":     ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
           "Nord":         ["AC", "AM", "AP", "PA", "RO", "RR", "TO"]}

# 2) Jointures : commande -> vendeur (1er article) -> état du vendeur ; commande -> état du client
vendeur_commande = items.sort_values("order_item_id").drop_duplicates("order_id")[["order_id", "seller_id"]]
routes = (livrees.merge(customers[["customer_id", "customer_state"]], on="customer_id")
                 .merge(vendeur_commande, on="order_id")
                 .merge(sellers[["seller_id", "seller_state"]], on="seller_id"))

# 3) Les 5 états vendeurs les plus importants, les autres regroupés
top_vendeurs = routes["seller_state"].value_counts().head(5).index
routes["origine"] = routes["seller_state"].where(routes["seller_state"].isin(top_vendeurs), "Autres")

# 4) Taux de retard et nombre de commandes pour chaque route
g = routes.groupby(["origine", "customer_state"]).agg(commandes=("order_id", "size"),
                                                      taux_retard=("en_retard", "mean")).reset_index()
g["taux_retard"] = g["taux_retard"] * 100

ordre_etats   = [e for r in REGIONS for e in REGIONS[r]]           # états clients rangés par région
ordre_origine = ["SP", "MG", "PR", "RJ", "SC", "Autres"]
taux = g.pivot(index="origine", columns="customer_state", values="taux_retard").reindex(index=ordre_origine, columns=ordre_etats)
nb   = g.pivot(index="origine", columns="customer_state", values="commandes").reindex(index=ordre_origine, columns=ordre_etats).fillna(0)
taux = taux.where(nb >= 30)                                         # moins de 30 commandes : case vide (pas fiable)

# 5) Heatmap
fig = go.Figure(go.Heatmap(
    z=taux.values, x=taux.columns, y=taux.index, customdata=nb.values,
    colorscale=[[0, "#FFFFFF"], [0.25, "#F3D6D2"], [1, ROUGE]], zmin=0, zmax=25, xgap=2, ygap=2,
    colorbar=dict(title=dict(text="% en retard", side="right"), ticksuffix=" %", thickness=10, len=0.8),
    text=[["" if np.isnan(v) else f"{v:.0f}" for v in ligne] for ligne in taux.values],
    texttemplate="%{text}", textfont=dict(size=10),
    hovertemplate="Vendeur %{y} → client %{x}<br>%{z:.1f} % en retard<br>%{customdata:.0f} commandes<extra></extra>"))

# Noms des régions au-dessus des colonnes, et un trait entre chaque région
position = 0
for region, etats in REGIONS.items():
    fig.add_annotation(x=position + (len(etats) - 1) / 2, y=1.06, yref="paper", showarrow=False,
                       text=f"<b>{region}</b>", font=dict(size=11, color=GRIS_FONCE))
    if position > 0:
        fig.add_vline(x=position - 0.5, line=dict(color=GRIS_FONCE, width=1.5))
    position += len(etats)

fig.update_xaxes(showline=False, ticks="", title=dict(text="État du client (regroupés par région)", font=dict(size=12)))
fig.update_yaxes(autorange="reversed", showline=False, ticks="", title=dict(text="État du vendeur", font=dict(size=12)))
mise_en_forme(fig, "Le Nord-Est est en retard, quel que soit le vendeur",
              "Part des commandes livrées en retard, selon l'état du vendeur et l'état du client · "
              "case vide : moins de 30 commandes", hauteur=430)
fig.update_layout(margin=dict(t=120, l=70, r=30, b=60))
fig.show()

In [13]:
# CALCULS SUPPLÉMENTAIRES : distance vendeur -> client (km) et montant de la commande (R$)

# 1) Nettoyer la géolocalisation : coordonnées au Brésil, puis UNE position moyenne par code postal
geo_bresil = geo[geo["geolocation_lat"].between(-34, 6) & geo["geolocation_lng"].between(-74, -34)]
geo_cp = geo_bresil.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
geo_cp.columns = ["lat", "lon"]
print("Codes postaux avec une position :", len(geo_cp))

# 2) Ajouter les codes postaux du client et du vendeur, puis leurs positions
routes = (routes.merge(customers[["customer_id", "customer_zip_code_prefix"]], on="customer_id")
                .merge(sellers[["seller_id", "seller_zip_code_prefix"]], on="seller_id")
                .merge(geo_cp.add_suffix("_client"),  left_on="customer_zip_code_prefix", right_index=True, how="left")
                .merge(geo_cp.add_suffix("_vendeur"), left_on="seller_zip_code_prefix",   right_index=True, how="left"))

# 3) Distance « à vol d'oiseau » avec la formule de haversine (la Terre est une sphère de rayon 6 371 km)
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

routes["distance_km"] = haversine_km(routes["lat_vendeur"], routes["lon_vendeur"],
                                     routes["lat_client"], routes["lon_client"])

# 4) Montant de chaque commande = prix des produits + frais de port
montants = items.groupby("order_id").agg(prix=("price", "sum"), fret=("freight_value", "sum"))
routes = routes.merge(montants, on="order_id", how="left")
routes["montant"] = routes["prix"] + routes["fret"]

# 5) Région du client (pour Q5)
ETAT_REGION = {etat: region for region, etats in REGIONS.items() for etat in etats}
routes["region_client"] = routes["customer_state"].map(ETAT_REGION)

# Résumé
ca_total  = routes["montant"].sum()
ca_retard = routes.loc[routes["en_retard"], "montant"].sum()
print(f"Distance médiane vendeur -> client : {fr(routes['distance_km'].median(), 0)} km")
print(f"Chiffre d'affaires total           : {fr(ca_total / 1e6)} M R$")
print(f"Dont commandes en retard           : {fr(ca_retard / 1e6)} M R$ ({fr(ca_retard / ca_total * 100)} %)")
print(f"Frais de port / prix des produits  : {fr(routes['fret'].sum() / routes['prix'].sum() * 100)} %")

Codes postaux avec une position : 19010
Distance médiane vendeur -> client : 434 km
Chiffre d'affaires total           : 15,3 M R$
Dont commandes en retard           : 1,2 M R$ (7,5 %)
Frais de port / prix des produits  : 16,6 %


In [14]:
# Q5 : préparation (vendeur) ou transport ? Délai moyen par région du client
par_region = routes.groupby("region_client").agg(
    preparation=("delai_preparation_j", "mean"),
    transport=("delai_transport_j", "mean"),
    taux_retard=("en_retard", "mean"),
    commandes=("order_id", "size"))
par_region["taux_retard"] = par_region["taux_retard"] * 100
par_region = par_region.sort_values("transport")            # la région la plus lente en haut

fig = go.Figure()
fig.add_trace(go.Bar(y=par_region.index, x=par_region["preparation"], orientation="h",
                     name="Préparation vendeur", marker_color=GRIS,
                     text=[f"{fr(v)} j" for v in par_region["preparation"]], textposition="inside",
                     insidetextanchor="middle", textfont=dict(color="white"),
                     hovertemplate="%{y}<br>Préparation : %{x:.1f} j<extra></extra>"))
fig.add_trace(go.Bar(y=par_region.index, x=par_region["transport"], orientation="h",
                     name="Transport", marker_color=ROUGE,
                     text=[f"{fr(v)} j" for v in par_region["transport"]], textposition="inside",
                     insidetextanchor="middle", textfont=dict(color="white"),
                     hovertemplate="%{y}<br>Transport : %{x:.1f} j<extra></extra>"))

# Au bout de chaque barre : délai total et taux de retard
for region, ligne in par_region.iterrows():
    total = ligne["preparation"] + ligne["transport"]
    fig.add_annotation(x=total, y=region, xshift=8, xanchor="left", showarrow=False,
                       font=dict(size=12, color=GRIS_FONCE),
                       text=f"= {fr(total)} j  ·  <span style='color:{ROUGE}'>{fr(ligne['taux_retard'], 0)} % en retard</span>")

fig.update_layout(barmode="stack", bargap=0.35)
fig.update_xaxes(title=dict(text="Délai moyen (jours)", font=dict(size=12)), range=[0, 33])
fig.update_yaxes(showline=False, ticks="", tickfont=dict(size=13))
mise_en_forme(fig, "La préparation dure 3 jours partout : c'est le transport qui fait la différence",
              "Délai moyen entre l'achat et la livraison, par région du client", hauteur=420)
fig.update_layout(showlegend=True, legend=dict(orientation="h", x=0, y=1.08, title=None, traceorder="normal"))
fig.show()

In [16]:
# Q6 : la promesse de délai est-elle réaliste ? (un point = un état du client)

# 1) Indicateurs par état du client
etats = routes.groupby("customer_state").agg(
    region=("region_client", "first"),
    taux_retard=("en_retard", "mean"),
    promis=("delai_promis_j", "mean"),          # délai promis au client
    reel=("delai_livraison_j", "mean"),         # délai réel
    ca=("montant", "sum"),                      # chiffre d'affaires (R$)
    commandes=("order_id", "size"))
etats["taux_retard"] = etats["taux_retard"] * 100
etats["marge"] = etats["promis"] - etats["reel"]          # NOUVEAU CALCUL : la « marge de sécurité » de la promesse
etats = etats[etats["commandes"] >= 30]                   # états avec assez de commandes

print("Corrélation marge / retard :", round(etats["marge"].corr(etats["taux_retard"]), 2))

# 2) Taille des bulles = chiffre d'affaires ; couleur = région
etats["taille"]  = 10 + 60 * np.sqrt(etats["ca"] / etats["ca"].max())
etats["couleur"] = np.where(etats["region"] == "Nord-Est", ROUGE,
                   np.where(etats["region"] == "Nord", BLEU, GRIS))

# Position des étiquettes pour éviter les chevauchements (comme au TP1)
positions = {"SP": "middle left", "MG": "middle left", "PR": "bottom center", "ES": "middle left",
             "AL": "middle right", "MA": "middle right", "SE": "middle right", "CE": "middle left",
             "BA": "middle left", "PB": "top right", "RN": "middle right", "PE": "bottom center",
             "AM": "bottom left", "AP": "bottom center"}
etats["pos"] = [positions.get(e, "top center") for e in etats.index]

fig = go.Figure(go.Scatter(
    x=etats["marge"], y=etats["taux_retard"], mode="markers+text",
    text=etats.index, textposition=etats["pos"], textfont=dict(size=11, color=GRIS_FONCE),
    marker=dict(size=etats["taille"], color=etats["couleur"], opacity=0.8, line=dict(color="white", width=1)),
    customdata=np.stack([etats["region"], etats["promis"], etats["reel"], etats["ca"] / 1e6], axis=1),
    hovertemplate="<b>%{text}</b> (%{customdata[0]})<br>promis %{customdata[1]:.1f} j · livré en %{customdata[2]:.1f} j"
                  "<br>marge %{x:.1f} j<br>%{y:.1f} % en retard<br>CA %{customdata[3]:.2f} M R$<extra></extra>"))

fig.add_annotation(x=8.3, y=23.5, xanchor="left", align="left", showarrow=False, font=dict(size=12, color=ROUGE),
                   text="<b>Nord-Est</b> : on promet<br>seulement 8 à 10 jours de marge")
fig.add_annotation(x=19.5, y=8, xanchor="right", align="right", showarrow=False, font=dict(size=12, color=BLEU),
                   text="<b>Nord</b> : très loin, mais<br>19 jours de marge → peu de retard")

fig.update_xaxes(title=dict(text="Marge de la promesse : délai promis − délai réel (jours)", font=dict(size=12)), range=[7, 21])
fig.update_yaxes(title=dict(text="% de commandes en retard", font=dict(size=12)), ticksuffix=" %", range=[0, 25])
mise_en_forme(fig, "Moins on prévoit de marge dans la promesse, plus on livre en retard",
              "Un point = un état du client (taille = chiffre d'affaires) · rouge : Nord-Est · bleu : Nord", hauteur=520)
fig.show()

Corrélation marge / retard : -0.69


In [17]:
# Q7 : par où commencer ? Les états qui comptent le plus de commandes en retard

NOMS_ETATS = {"SP": "São Paulo", "RJ": "Rio de Janeiro", "MG": "Minas Gerais", "BA": "Bahia",
              "RS": "Rio Grande do Sul", "SC": "Santa Catarina", "ES": "Espírito Santo", "PR": "Paraná"}

# 1) Indicateurs par état du client
prio = routes.groupby("customer_state").agg(
    region=("region_client", "first"),
    commandes=("order_id", "size"),
    en_retard=("en_retard", "sum"),                 # NOMBRE de commandes en retard
    taux=("en_retard", "mean"),                     # TAUX de retard
    note=("review_score", "mean"))
prio["taux"] = prio["taux"] * 100
prio["ca_retard"] = routes[routes["en_retard"]].groupby("customer_state")["montant"].sum()

# 2) Classer par nombre de commandes en retard (l'impact réel), garder les 8 premiers
prio = prio.sort_values("en_retard", ascending=False).head(8).reset_index()
prio["part"]  = prio["en_retard"] / routes["en_retard"].sum() * 100        # part de tous les retards
prio["cumul"] = prio["part"].cumsum()                                       # part cumulée

# 3) Le taux en rouge quand il dépasse 10 %
fond_taux    = ["#F3D6D2" if v > 10 else "white" for v in prio["taux"]]
couleur_taux = [ROUGE if v > 10 else GRIS_FONCE for v in prio["taux"]]
n = len(prio)

fig = go.Figure(go.Table(
    columnwidth=[28, 150, 85, 80, 80, 85, 70, 60, 85, 65],
    header=dict(values=["<b>#</b>", "<b>État du client</b>", "<b>Région</b>", "<b>Commandes</b>", "<b>% en retard</b>",
                        "<b>Commandes<br>en retard</b>", "<b>Part des<br>retards</b>", "<b>Cumul</b>",
                        "<b>CA en retard</b>", "<b>Note<br>moyenne</b>"],
                fill_color="#F3F4F6", line_color="#E5E7EB", height=40,
                align=["center", "left", "left"] + ["right"] * 7, font=dict(size=12, color="#1F2937")),
    cells=dict(values=[list(range(1, n + 1)),
                       [f"<b>{NOMS_ETATS[e]}</b> ({e})" for e in prio["customer_state"]],
                       prio["region"],
                       [fr(v, 0) for v in prio["commandes"]],
                       [f"<b>{fr(v)} %</b>" for v in prio["taux"]],
                       [fr(v, 0) for v in prio["en_retard"]],
                       [f"{fr(v, 0)} %" for v in prio["part"]],
                       [f"{fr(v, 0)} %" for v in prio["cumul"]],
                       [f"{fr(v / 1000, 0)} k R$" for v in prio["ca_retard"]],
                       [fr(v) for v in prio["note"]]],
               fill_color=[["white"] * n] * 4 + [fond_taux] + [["white"] * n] * 5,
               font=dict(size=12, color=[[GRIS_FONCE] * n] * 4 + [couleur_taux] + [[GRIS_FONCE] * n] * 5),
               line_color="#E5E7EB", height=32, align=["center", "left", "left"] + ["right"] * 7)))

mise_en_forme(fig, "São Paulo et Rio concentrent la moitié des commandes en retard",
              "Les 8 états qui comptent le plus de commandes en retard (80 % du total) · en rouge : plus de 10 % de retard",
              hauteur=420)
fig.update_layout(margin=dict(t=95, l=20, r=20, b=10))
fig.show()

In [18]:
# EXPORT : on sauvegarde les commandes préparées pour la carte (TP2) et la base (TP4)
colonnes = ["order_id", "order_purchase_timestamp", "customer_state", "region_client",
            "seller_id", "seller_state", "lat_client", "lon_client", "lat_vendeur", "lon_vendeur",
            "distance_km", "delai_livraison_j", "delai_preparation_j", "delai_transport_j",
            "delai_promis_j", "retard_j", "en_retard", "review_score", "montant"]
routes[colonnes].to_pickle("data/commandes.pkl")
print("Sauvegardé :", len(routes), "commandes,", len(colonnes), "colonnes")

Sauvegardé : 96018 commandes, 19 colonnes
